# V07 — Seaborn & Matplotlib

**When to use Seaborn/Matplotlib instead of Plotly:**
- Statistical plots with built-in inference (confidence bands, regression diagnostics)
- Publication-quality static figures for reports
- Pairplots, FacetGrids, residual plots
- Fine-grained `axes`-level control for academic/journal style

**Mental model:** Matplotlib is the canvas. Seaborn is a higher-level API built on top. Most seaborn functions return an `Axes` object you can further customize with matplotlib.

**Reference:** [Seaborn docs](https://seaborn.pydata.org/) | [Matplotlib docs](https://matplotlib.org/stable/)

**Allowed:** `seaborn`, `matplotlib`, `pandas`, `numpy`, `scipy.stats`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_openml, fetch_california_housing
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# --- Datasets ---
housing_raw = fetch_california_housing(as_frame=True)
housing = housing_raw.frame.copy()
housing.columns = [c.lower() for c in housing.columns]
housing['price_tier'] = pd.qcut(housing['medhousval'], q=4, labels=['Q1','Q2','Q3','Q4'])

credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = credit_raw.copy()
credit['credit_amount'] = pd.to_numeric(credit['credit_amount'], errors='coerce')
credit['duration'] = pd.to_numeric(credit['duration'], errors='coerce')
credit['age'] = pd.to_numeric(credit['age'], errors='coerce')

retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['Month'] = retail['InvoiceDate'].dt.to_period('M').astype(str)

print(f"Housing: {housing.shape} | Credit: {credit.shape}")

---
## Exercise 1 — Seaborn Figure-Level vs Axes-Level

**Concept:** Seaborn has two types of functions:
- **Figure-level** (`sns.relplot`, `sns.displot`, `sns.catplot`): manage their own `Figure` and support `col`/`row` faceting
- **Axes-level** (`sns.scatterplot`, `sns.histplot`, `sns.boxplot`): draw on a specific `ax`

**Spec:** Build the same chart both ways.

1. Use **figure-level** `sns.displot` to plot `medhousval` distribution faceted by `price_tier` (4 panels in a row)
   - `kind='kde'`, `fill=True`, `col='price_tier'`, `col_wrap=4`
   - Title the overall figure: `'House Value Distribution by Price Tier (Figure-Level)'`

2. Use **axes-level** `sns.kdeplot` to overlay all 4 tiers on a single axes inside a manually created `plt.subplots` figure
   - One color per tier, `fill=True`, `alpha=0.3`
   - Add legend with tier labels
   - Title: `'House Value Distribution by Price Tier (Axes-Level)'`

3. Assign the axes-level figure to `fig_ax1` and axes to `ax1`

In [ ]:
# YOUR CODE HERE
fig_ax1, ax1 = None, None
plt.tight_layout()
plt.show()

In [ ]:
# --- ASSERTIONS ---
assert fig_ax1 is not None and ax1 is not None
assert isinstance(fig_ax1, matplotlib.figure.Figure)
assert ax1.get_title() == 'House Value Distribution by Price Tier (Axes-Level)'
# 4 KDE lines on the axes (one per tier)
n_lines = len([l for l in ax1.lines if len(l.get_xdata()) > 1])
assert n_lines >= 4, f"Expected 4 KDE lines, got {n_lines}"
assert ax1.get_legend() is not None, "Legend required"
print("✓ Exercise 1 passed")

---
## Exercise 2 — Statistical Regression Plot

**Spec:** Regression plot with confidence band and residual diagnostics.

1. Create a `1×2` subplot figure, `figsize=(14, 5)`
2. Left panel: `sns.regplot` of `medinc` vs `medhousval`
   - `scatter_kws={'alpha':0.1, 's':10, 'color':'#1565C0'}`
   - `line_kws={'color':'#E53935', 'linewidth':2}`
   - `ci=95`
   - Title: `'Linear Regression: Income vs House Value'`
3. Right panel: residual plot using `sns.residplot`
   - Same variables
   - `lowess=True`
   - `scatter_kws={'alpha':0.1, 's':10}`
   - Add horizontal line at y=0: `ax.axhline(0, color='red', linestyle='--')`
   - Title: `'Residuals'`
4. Assign figure to `fig2`

In [ ]:
# YOUR CODE HERE
fig2 = None
plt.tight_layout()
plt.show()

In [ ]:
# --- ASSERTIONS ---
assert isinstance(fig2, matplotlib.figure.Figure)
axes = fig2.get_axes()
assert len(axes) == 2, "Must have 2 subplots"
assert axes[0].get_title() == 'Linear Regression: Income vs House Value'
assert axes[1].get_title() == 'Residuals'
# Residual plot should have horizontal line
hlines = [l for l in axes[1].lines if len(set(l.get_ydata())) == 1]
assert len(hlines) >= 1, "Horizontal reference line at y=0 required in residual plot"
print("✓ Exercise 2 passed")

---
## Exercise 3 — Pairplot with Custom Diagonal & Annotations

**Spec:** Pairplot is Seaborn's most powerful EDA function — but needs customization to be useful.

1. Use `sns.pairplot` on a 1000-row sample of housing with columns `['medinc','houseage','averooms','medhousval']`
   - `hue='price_tier'`, `diag_kind='kde'`
   - `plot_kws={'alpha':0.3, 's':15}`
   - `palette='Set2'`
2. After creating, iterate over the lower triangle axes and add Pearson r annotation:
   - For each off-diagonal ax in `g.axes`, compute and display `r=X.XX` in the top-left corner
3. Set the figure title: `'Housing Pairplot with Correlation Annotations'`
4. Assign the `PairGrid` to `g3`

In [ ]:
housing_sample = housing[['medinc','houseage','averooms','medhousval','price_tier']].sample(1000, random_state=42)
pairplot_cols = ['medinc','houseage','averooms','medhousval']

# YOUR CODE HERE
g3 = None
plt.show()

In [ ]:
# --- ASSERTIONS ---
assert g3 is not None
assert isinstance(g3, sns.PairGrid)
assert g3.figure is not None
n_cols = len(pairplot_cols)
assert g3.axes.shape == (n_cols, n_cols)
# Check that off-diagonal axes have text annotations (r= values)
annotated = 0
for i in range(n_cols):
    for j in range(n_cols):
        if i != j:
            texts = g3.axes[i][j].texts
            if any('r=' in t.get_text() for t in texts):
                annotated += 1
assert annotated > 0, "Must annotate at least some off-diagonal panels with r="
print(f"✓ Exercise 3 passed — {annotated} panels annotated with Pearson r")

---
## Exercise 4 — FacetGrid: Custom Multi-Panel Plots

**Spec:** `FacetGrid` lets you map any plot function across facets — more flexible than `catplot`.

1. Create a `sns.FacetGrid` on the credit dataset, `col='class'`, `row=None`, `height=4`, `aspect=1.2`
2. Map `sns.histplot` onto it for `credit_amount` with `bins=30`, `kde=True`
3. Set x-axis labels: `'Credit Amount (DM)'`
4. Add a vertical line at the mean for each facet using `g.map` with a custom function
5. Set overall title: `'Credit Amount Distribution by Credit Class'`
6. Assign to `g4`

In [ ]:
def vline_mean(x, **kwargs):
    """Custom function to add a mean vertical line."""
    plt.axvline(x.mean(), color='red', linestyle='--', linewidth=1.5,
                label=f'Mean: {x.mean():,.0f}')

# YOUR CODE HERE
g4 = None
plt.show()

In [ ]:
# --- ASSERTIONS ---
assert g4 is not None
assert isinstance(g4, sns.FacetGrid)
assert g4.axes.shape[1] == 2, "Must have 2 columns (good and bad credit)"
# Each panel should have a vertical line (mean)
for ax in g4.axes.flat:
    vlines = [l for l in ax.lines if len(set(l.get_xdata())) == 1]
    assert len(vlines) >= 1, f"Mean vline missing from panel '{ax.get_title()}'"
print("✓ Exercise 4 passed")

---
## Exercise 5 — Clustermap: Hierarchical Heatmap

**Spec:** `clustermap` clusters both rows and columns — revealing natural groupings in data.

1. Build a pivot table: mean `credit_amount` by `purpose` (rows) × `employment` (columns)
2. Use `sns.clustermap`:
   - `cmap='YlOrRd'`, `annot=True`, `fmt='.0f'`
   - `figsize=(10, 8)`
   - `method='ward'` (hierarchical clustering method)
   - `standard_scale=1` (normalize columns to 0-1 range)
   - `linewidths=0.5`
3. Add title: `'Credit Amount Heatmap: Purpose × Employment (Ward Clustering)'`
4. Assign the `ClusterGrid` to `g5`

In [ ]:
pivot = credit.pivot_table(
    values='credit_amount', index='purpose', columns='employment', aggfunc='mean'
).round(0)

# YOUR CODE HERE
g5 = None
plt.show()

In [ ]:
# --- ASSERTIONS ---
assert g5 is not None
assert isinstance(g5, sns.matrix.ClusterGrid)
# Check dendrogram present (clustering was applied)
assert g5.ax_row_dendrogram is not None
assert g5.ax_col_dendrogram is not None
# Check annotations present
annotations = [t for t in g5.ax_heatmap.texts]
assert len(annotations) > 0, "Must annotate cells with values"
print("✓ Exercise 5 passed")

---
## Exercise 6 — Matplotlib: Full Figure Composition

**Spec:** Build a complex figure using matplotlib's `GridSpec` for irregular subplot layouts — something neither Seaborn nor Plotly subplots handles as flexibly.

Layout using `gridspec.GridSpec(3, 3)`:
- Top-left (2×2 cells): Large scatter of `medinc` vs `medhousval`, colored by `price_tier`
- Top-right (1×1): Histogram of `medhousval`
- Middle-right (1×1): Histogram of `medinc`
- Bottom row (3×1): Three violin plots — `medhousval` per `price_tier`

Style:
- Use `sns.set_theme(style='whitegrid')`
- Overall figure title: `'Housing Market Composition View'`, fontsize 14
- `figsize=(12, 10)`
- Assign figure to `fig6`

In [ ]:
import matplotlib.gridspec as gridspec
sns.set_theme(style='whitegrid')

# YOUR CODE HERE
fig6 = None

plt.tight_layout()
plt.show()

In [ ]:
# --- ASSERTIONS ---
assert isinstance(fig6, matplotlib.figure.Figure)
assert fig6.get_size_inches().tolist() == [12.0, 10.0]
axes = fig6.get_axes()
# Should have: 1 scatter + 2 histograms + 3 violins = 6 axes (approx)
assert len(axes) >= 5, f"Expected at least 5 axes, got {len(axes)}"
# Check figure-level title
titles = [t.get_text() for t in fig6.texts]
assert any('Housing Market' in t for t in titles), "Figure title missing"
print("✓ Exercise 6 passed")

---
## Exercise 7 — When Seaborn Wins: Statistical Plot Showcase

**Spec:** This exercise demonstrates 4 charts that Seaborn does better than Plotly for statistical analysis.

Build a `2×2` subplot figure, `figsize=(14, 10)`:

1. **Top-left:** `sns.ecdfplot` of `credit_amount` for each credit class — cumulative distribution
   - Both classes on same axes, `hue='class'`, `palette=['#43A047','#E53935']`
   - Title: `'ECDF: Credit Amount by Class'`

2. **Top-right:** `sns.violinplot` of `credit_amount` by `purpose` (top 5 by count)
   - `hue='class'`, `split=True` (half violin per class)
   - `inner='quartile'`, `palette=['#43A047','#E53935']`
   - Title: `'Split Violin: Credit Amount by Purpose'`

3. **Bottom-left:** `sns.heatmap` of correlation matrix (housing)
   - Masked upper triangle, `annot=True`, `cmap='coolwarm'`, `center=0`
   - Title: `'Correlation Matrix'`

4. **Bottom-right:** `sns.lmplot`-style scatter using `sns.regplot` per class on same axes
   - `duration` vs `credit_amount`, separate regression lines per class
   - Title: `'Regression by Credit Class'`

Assign figure to `fig7`

In [ ]:
top5_purpose = credit['purpose'].value_counts().nlargest(5).index
credit_top5 = credit[credit['purpose'].isin(top5_purpose)].copy()

feat_cols = ['medinc','houseage','averooms','avebedrms','population','aveoccup','medhousval']
corr_matrix = housing[feat_cols].corr()
mask_upper = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

# YOUR CODE HERE
fig7, axes7 = None, None

plt.tight_layout()
plt.show()

In [ ]:
# --- ASSERTIONS ---
assert isinstance(fig7, matplotlib.figure.Figure)
axes_list = fig7.get_axes()
assert len(axes_list) >= 4
titles = [ax.get_title() for ax in axes_list]
assert 'ECDF: Credit Amount by Class' in titles
assert 'Correlation Matrix' in titles
assert 'Regression by Credit Class' in titles
print("✓ Exercise 7 passed")

---
## Exercise 8 — Capstone: Publication-Quality Report Figure

**Spec:** Build a single matplotlib figure suitable for a consulting report or academic paper — the use case where Seaborn/Matplotlib still beats Plotly.

**Subject:** A complete credit risk analysis summary.

Using `plt.style.use('seaborn-v0_8-paper')` and `figsize=(16, 12)`, build a 3×3 subplot grid showing:

Row 1:
1. Credit amount distribution (histogram + KDE) by class
2. Duration distribution by class (overlapping KDE, filled)
3. Age distribution by class

Row 2:
4. Correlation heatmap of numeric features
5. Scatter: duration vs credit_amount colored by class, with regression lines
6. Box plot: credit_amount by employment, colored by class

Row 3 (full-width): Grouped bar of default rate by purpose (span all 3 columns using `subplot2grid`)

Requirements:
- Every panel has a title and axis labels
- Consistent color scheme: good=`'#43A047'`, bad=`'#E53935'`
- Figure suptitle: `'Credit Risk Analysis Report'`, fontsize 16
- Assign to `fig8`

In [ ]:
try:
    plt.style.use('seaborn-v0_8-paper')
except:
    plt.style.use('seaborn-paper')

GOOD_COLOR = '#43A047'
BAD_COLOR = '#E53935'

default_rate_by_purpose = (
    credit.assign(is_bad=(credit['class']=='bad').astype(int))
    .groupby('purpose')['is_bad'].mean()
    .sort_values(ascending=False)
)

# YOUR CODE HERE
fig8 = None

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# --- ASSERTIONS ---
assert isinstance(fig8, matplotlib.figure.Figure)
assert fig8.get_size_inches().tolist() == [16.0, 12.0]
# Suptitle
suptitle_texts = [t.get_text() for t in fig8.texts]
assert any('Credit Risk Analysis Report' in t for t in suptitle_texts), "Suptitle missing"
# At least 7 axes (6 individual panels + 1 full-width bar)
axes = fig8.get_axes()
assert len(axes) >= 7, f"Expected >= 7 axes, got {len(axes)}"
# Check titles on panels
titles = [ax.get_title() for ax in axes if ax.get_title()]
assert len(titles) >= 6, f"Expected at least 6 panel titles, got {len(titles)}"
print(f"✓ Exercise 8 passed — {len(axes)} axes, {len(titles)} titles")